# Anomalies in the empty container depot

The following data comes from a project and should be treated confidentially.
This is explicitly *not* [open data](https://www.bpb.de/gesellschaft/medien/opendata/64055/was-sind-offene-daten?p=all) and you should assume that [copyright applies](https://www.forschungsdaten.info/themen/rechte-und-pflichten/gesetzliche-regelungen-im-ueberblick/).
You are hereby granted permission to use the data as part of the exercise.
You are not permitted to carry out any other actions, such as archiving or distributing the data.

In [ ]:
#raise Exception("By removing these lines, I signal that I have understood, "
                #"that the data must be treated confidentially")

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from pandas.plotting import autocorrelation_plot

pd.options.display.max_rows = 10  # display a maximum of 10 entries of a DataFrame

## Reading the data

### Right to use data

You have received data from a research project.
This is explicitly not open data.

<span style="color:blue">
    (a) Which right probably applies?
</span> 

**Answer:** ...

<span style="color:blue"> (b) Which license would restrict the use of the data less?
</span>

**Answer:** ...

After the legal issues have been clarified, we read in the data.

In [ ]:
df = pd.read_csv("Leercontainerdepot_extract.csv", delimiter=";")

eingang = pd.to_datetime(df.Eingang,format="mixed", dayfirst=True)  # date in csv looks like "01.10.2014 06:03"
df = df.assign(Eingang=eingang)

ausgang = pd.to_datetime(df.Ausgang, dayfirst=True, errors='coerce')  # sometimes seemingly random, e.g. '41943,65758'
df = df.assign(Ausgang=ausgang)

The parameter `dayfirst` is due to the US date format.
Even with the dot notation `DD.MM.YYYY` common in Germany, the first element is interpreted as the month by default, i.e. as `MM.DD.YYYY`.
This can be prevented, for example, by using ISO-8601-compliant time formatting.

After reading in, you can use `df.info()` to see whether everything worked successfully.

In [ ]:
df.info()
if 'datetime_column' in df.select_dtypes(include=['datetime']):
    df['datetime_column'] = df['datetime_column'].view('int64')  # Convert to numeric
summary = df.describe()
df

### The first impression

<span style="color:blue">
    (a)For which attributes were all values ​​successfully read in?
</span> 

**Answer:** ...

<span style="color:blue"> (b) What does it mean when pandas assigns the type `object` to a column?
</span>

**Answer:** ...

<span style="color:blue"> (c) Which attributes had problems reading and are missing entries?
How does this relate to the parameter `errors='coerce'`?
</span>

**Answer:** ...

<span style="color:blue"> (d) What content-related reasons could there be for entries to be missing in this case?
</span>

**Answer:** ...

<span style="color:blue"> (e) What time period does the data come from?
</span>

**Answer:** ...

## Visual inspection of the data set

### Entries

When empty containers are brought onto the site, this is referred to as an entry.
The containers are then stowed away and repaired or cleaned if necessary.

#### Resampling of data

Resampling is carried out in preparation for the following graphics.
A more detailed description of how resampling works is available at https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.resample.html

Here, resampling is carried out using the parameter `'H'`.

In [ ]:
df_eingang = df.set_index("Eingang")
series_eingang_hourly = df_eingang["Ladungsart"].resample('h').count()

<span style="color:blue"> (a) What exactly does the expression  `df_eingang["Ladungsart"].resample('h')` do?
</span>

**Answer:** ...

In [ ]:
# You are welcome to analyze 'series_eingang_hourly' here to answer the question

# your Code...

<span style="color:blue"> (b) Which values ​​other than `'h'` could also make sense when resampling?
Find three examples.
</span>

**Answer:** ...

Now that this has been clarified, the following part is much easier to understand.

#### Descriptive Statistics

In [ ]:
series_eingang_hourly.describe()

<span style="color:blue"> (a) How should the descriptive statistics of the inputs be interpreted?
How should the hours for which no inputs were recorded be understood??
</span>

**Answer:** ...

<span style="color:blue"> (b) What explains the longer gap right at the beginning?
</span>

**Answer:** ...

<span style="color:blue"> (c) Which periods should be excluded from this time series in order to only look at the actual operational process?
 Create a list of explicit instructions.
</span>

**Answer:** ...

In [ ]:
series_eingang_hourly.plot(figsize=(20, 5))
plt.show()

In the following, Saturday and Sunday are removed.
In pandas, the days of the week are encoded with the numbers 0 to 6, see [Documentation](https://pandas.pydata.org/pandas-docs/version/0.22/generated/pandas.DatetimeIndex.dayofweek.html).
A mask is used for this.

In [ ]:
mask = series_eingang_hourly.index.dayofweek < 5
mask

This mask can now be used to only look at the days during the week.
The time index remains initially, i.e. the values ​​on the weekend are deleted.
Nevertheless, as long as there is a time index, the graphic looks the same because the days still exist in the index.
To prevent this, the index must be set to a numerical value that only represents a chronological order and no longer a date.

In [ ]:
series_eingang_hourly_only_business_days_with_index = series_eingang_hourly[mask]
print(series_eingang_hourly_only_business_days_with_index.describe())
series_eingang_hourly_only_business_days = series_eingang_hourly_only_business_days_with_index.reset_index(drop=True)
series_eingang_hourly_only_business_days.plot(figsize=(20, 5))
plt.show()

How can the gap after the first two peaks be explained?

#### Everyday work in the form of box plots

In the following, the inputs for each hour are summarized in the form of box plots.

In [ ]:
series_eingang_hourly_only_business_days_with_index_as_df = pd.DataFrame(
    data=series_eingang_hourly_only_business_days_with_index)
eingang_grouped_by_hour = series_eingang_hourly_only_business_days_with_index_as_df.groupby(
    series_eingang_hourly_only_business_days_with_index_as_df.index.hour)
eingang_grouped_by_hour.boxplot(figsize=(24, 6), subplots=False)
plt.xticks(np.arange(25), np.arange(25))  # Time 00:00 bis 23:00
plt.xlabel("Time")
plt.ylabel("Number of inputs")
plt.show()

<span style="color:blue"> (a) Which time zone do you think the data comes from?
Is it suitable for summarizing the working days across time changes?
</span>

**Answer:** ...

<span style="color:blue"> (b) What could the opening hours of the company be?
 Also consider the temporal resolution of the x-axis.
</span>

**Answer:** ...

<span style="color:blue"> (c) Which anomalies can already be explained using the findings from the previous part- Time Series?
</span>

**Answer:** ...

#### Daily statistics

The values ​​for each day are summarized below.

In [ ]:
daily_sum = series_eingang_hourly_only_business_days_with_index.resample("D").sum()
daily_sum_only_open_days = daily_sum[daily_sum != 0]
print(daily_sum_only_open_days.describe())
daily_sum_only_open_days.plot(style=".")
plt.show()

<span style="color:blue"> (a) What expression was used to generate the mask that deletes the zero values ​​in the daily statistics?
</span>

**Answer:** ...

In [ ]:
daily_sum_only_open_days.plot.hist(bins=5)
plt.show()

<span style="color:blue"> (b) Can one speak of a normal distribution for the daily inputs after visual inspection?
    Why (not)?
</span>

**Answer:** ...

The accepted maximum and minimum are determined below.

In [ ]:
factor = 1.96

global_average = daily_sum_only_open_days.mean()
standard_deviation = daily_sum_only_open_days.std()
bound_of_nintey_percent_interval = factor * standard_deviation
acceptable_minimum = global_average - bound_of_nintey_percent_interval
acceptable_maximum = global_average + bound_of_nintey_percent_interval
acceptable_minimum, acceptable_maximum

<span style="color:blue"> (c) What values ​​are typically used for the variable `factor`?
 What value must you set it to in order to get a confidence interval of 90%, 95% or 99%?
</span>

**Answer:** ...

These acceptable minimum and maximum can now be used in the form of masks.

In [ ]:
mask1 = daily_sum_only_open_days < acceptable_minimum
mask2 = daily_sum_only_open_days > acceptable_maximum

Masks in pandas can be linked with a `&` or a `|`.
The `&` corresponds to an And, the `|` to an Or.

In [ ]:
mask = mask1  # <-- Replace the mask with a suitable variant
ausreisser = daily_sum_only_open_days[mask]
ausreisser.plot(style=".")
plt.show()

<span style="color:blue"> (d) How many outliers are there for each factor that is possible according to (c)?
</span>

**Answer:** ...

<span style="color:blue"> (e) Explain in your own words what a mask is.
 Create an explanatory sketch
 (the easiest way to do this is on the printed task sheet with a pen).
</span>

**Answer:** ...

### Outputs

Create a new notebook (or copy it and give it a descriptive name).
Perform the same analysis for the outputs as already done for the inputs.
Make sure you use descriptive, suitable variable names.

#### Event data
	
What do you notice when looking at the outputs as a DataFrame?

**Answer:** ...

#### Summary of a working day as boxplots 
	
What do you notice when looking at the boxplots of the outputs?

**Answer:** ...

#### Daily statistics

What do you notice when looking at the boxplots of the outputs?

**Answer:** ...

## Generated attribute: dwell time

The dwell time is the time between the output and the input

In [ ]:
verweildauer_df = pd.DataFrame(data=df.Ausgang - df.Eingang, columns=["Verweildauer"], index=df.index)
verweildauer_df.info()
verweildauer_df.describe()

<span style="color:blue"> How many entries are there for the dwell time? 
 What does the number of valid values ​​depend on?
</span>

**Answer:** ...

### Distribution of the dwell time

In the following the residence time of the empty containers is shown.

In [ ]:
def turn_nanoseconds_into_faction_of_days(x, pos):
    return "%.2f" % (x / 86400000000000)


ax = verweildauer_df.plot(style=".")
ax.yaxis.set_major_formatter(FuncFormatter(turn_nanoseconds_into_faction_of_days))
plt.ylabel("Days")
plt.title("Dwell time of the containers")
plt.show()

How many entries are there for the dwell time? 
What does the number of valid values depend on?

**Answer:** ...

<span style="color:blue"> (a)How should the x-axis in the plot be interpreted?
 What do the numbers 0 - 30,000 mean? 
 Why is an area missing?
</span>

**Answer:** ...

Masks can also be created 'on the fly' and used straight away.

In [ ]:
negative_zeiten = verweildauer_df[verweildauer_df.Verweildauer < pd.Timedelta(0)]
negative_zeiten.describe()

The index of a data frame or series can also be used with `iloc` to view corresponding entries in another data frame or series.

In [ ]:
df_mit_verweildauer = df.assign(Verweildauer=verweildauer_df.Verweildauer)
df_mit_verweildauer.iloc[negative_zeiten.index]

<span style="color:blue"> (b) What does the negative dwell time mean?
</span>

**Answer:** ...

<span style="color:blue"> (c) How many hours and minutes of negative dwell time does the minimum of `-1 days +10:19:00` correspond to?
</span>

**Answer:** ...

### Distribution of dwell time

Boxplots generally provide information about outliers.
Density plots help to better identify clusters.
There are also other forms of visualization to illuminate certain aspects of the distribution.

In [ ]:
verweildauer_df

In [ ]:
verweildauer_df["Verweildauer in Tagen"] = (verweildauer_df['Verweildauer'] / pd.Timedelta(days=1))

verweildauer_df

In [ ]:
verweildauer_df["Verweildauer in Tagen"].plot.kde()
plt.show()
plt.ylabel("Days")

verweildauer_df["Verweildauer in Tagen"].plot.box(vert=False)
plt.ylabel("Days")
plt.show()

verweildauer_df["Verweildauer in Tagen"].plot.kde(logy=True)
plt.ylabel("Days")
plt.show()

verweildauer_df["Verweildauer in Tagen"].plot.box(logx=True, vert=False)
plt.ylabel("Days")
plt.show()

<span style="color:blue"> (a)How do the plots with and without logarithmic scaling of the y-axis differ?
 Which variant do you prefer and why?
</span>

**Answer:** ...

<span style="color:blue"> (b) Can we speak of a normal distribution in terms of length of stay?
 Why (not)?
</span>

**Answer:** ...

## The attributes X1 and X2

### Interpretation of X1 and X2

<span style="color:blue">What could the attributes X1 and X2 mean?
How would you further process the attributes to generate added value from them?
</span>

**Answer:** ...

In [ ]:
# Your Code...

### Machine learning for prediction of dwell time

<span style="color:blue">Which machine learning methods could estimate the dwell time of the empty container from X1 and X2?
 What would be possible input parameters from the `DataFrame`?
 What exactly would be the output of the method?
 In which unit would you measure the output?
</span>

**Answer:** ...

In [ ]:
# Your Code...

Tip: In the conference paper
["A Dwell Time-based Container Positioning Decision Support System at a Port Terminal" by Gaete, González-Araya, González-Ramírez and Astudillo](https://www.researchgate.net/profile/Marcela_Gonzalez-Araya/publication/314522056_A_Dwell_Time-based_Container_Positioning_Decision_Support_System_at_a_Port_Terminal/links/58d31f9492851c319e56f30e/A-Dwell-Time-based-Container-Positioning-Decision-Support-System-at-a-Port-Terminal.pdf)
presents how to predict dwell time using machine learning methods.